In [ ]:
import os
import sys

from pyspark.sql import SparkSession



In [ ]:
# ==============================================================================
# PHASE 3: Reverse Sync to Central Catalog


In [ ]:
# ==============================================================================
print("Initializing Phase 3 Reverse Sync Pipeline...")

uc_table_path = "s3a://lakehouse-bucket/unity_catalog/transformed_accounts"

# 1. Initialize Spark session with Central Catalog configuration
print("\nInitializing Spark Session for Central Catalog Registration...")
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Reverse-Sync-To-Central-Catalog") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,org.apache.iceberg:iceberg-aws-bundle:1.5.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.central_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.central_catalog.catalog-impl", "org.apache.iceberg.rest.RESTCatalog") \
    .config("spark.sql.catalog.central_catalog.uri", "http://iceberg-rest:8181") \
    .config("spark.sql.catalog.central_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.central_catalog.warehouse", "s3a://lakehouse-bucket/central_warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localstack:4566") \
    .config("spark.sql.catalog.central_catalog.s3.endpoint", "http://localstack:4566") \
    .config("spark.sql.catalog.central_catalog.s3.path-style-access", "true") \
    .config("spark.sql.catalog.central_catalog.client.region", "us-east-1") \
    .config("spark.sql.catalog.central_catalog.s3.access-key-id", "test") \
    .config("spark.sql.catalog.central_catalog.s3.secret-access-key", "test") \
    .config("spark.hadoop.fs.s3a.access.key", "test") \
    .config("spark.hadoop.fs.s3a.secret.key", "test") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

# 2. Adopt Delta-written files into an Iceberg table
table_name = "default.transformed_accounts_synced"
print(f"\nRegistering Unity Catalog table to Central Catalog as: {table_name}")

try:
    # Drop if it already exists for idempotency
    spark.sql(f"DROP TABLE IF EXISTS central_catalog.{table_name}")
    
    print("Creating Iceberg table schema matching the Parquet files...")
    spark.sql(f"""
        CREATE TABLE central_catalog.{table_name}
        USING iceberg
        AS SELECT * FROM parquet.`{uc_table_path}` WHERE 1=0
    """)
    
    print("Adopting Parquet files using Iceberg add_files procedure...")
    spark.sql(f"""
        CALL central_catalog.system.add_files(
            table => 'central_catalog.{table_name}',
            source_table => '`parquet`.`{uc_table_path}`'
        )
    """)
    
    print("✅ Reverse Sync Complete! Table successfully populated in Central Catalog.")
except Exception as e:
    print(f"❌ Failed to sync table: {e}")
    import sys
    sys.exit(1)


In [ ]:
# ==============================================================================
# PHASE 4: AWS Consumer Simulator (Validation)


In [ ]:
# ==============================================================================
print("\n--- [Phase 4] Validating Central Catalog (Simulating AWS Athena) ---")
print(f"Executing Query: SELECT * FROM central_catalog.{table_name}")

try:
    df_synced = spark.table(f"central_catalog.{table_name}")
    print("\nSchema retrieved from Central Catalog:")
    df_synced.printSchema()
    
    print("\nData retrieved from Central Catalog:")
    df_synced.show()
    print("✅ Validation Successful! The Delta table created in Unity Catalog is fully queryable from the Central Catalog as an Iceberg table.")
except Exception as e:
    print(f"❌ Validation Failed: {e}")

spark.stop()
